# Notebook 9 — build $R_{ij}$ from the macroatom

Level-1 code: count where the macroatom's packets come out, $R_{ij} = N(i\to j)/\sum_k N(i\to k)$; throw the level network away and run with $R$ alone; compare ε*, $R$ and the macroatom.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom
from rtedu.transport import run, emergent_by_line
from rtedu.matrix import group_of_line, MacroatomRedistribution, build_R, MatrixRedistribution, low_rank, interpolate_R, row_error
from rtedu.redistribution import EpsilonRedistribution
from rtedu.bands import band_fluxes, magnitudes, colours
rng = np.random.default_rng(rtedu.SEEDS["ch09"])
atom = five_level_atom(); nm = 1e7 * atom.lam_cm
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nu_launch = atom.nu[0] * 1.001
def spectrum(model, n, seed):
    """emergent line spectrum, band magnitudes and colours of n blue packets under a redistribution model"""
    nu, last, n_int = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau, r_out, t, model)
    m = magnitudes(band_fluxes(nu)); return emergent_by_line(last, atom.n_lines), m, colours(m), float(n_int.mean())

## The reference: the macroatom in the transport

Every interaction activates the macroatom at the line's upper level (β from the line list); the walk names the exit line. We collect every (absorbed line, emitted line) event.

In [ ]:
n = 4000
macro = MacroatomRedistribution(atom, tau)
t0 = time.time(); spec_macro, mag_macro, col_macro, nint_macro = spectrum(macro, n, rtedu.SEEDS["ch09"] + 1); wall_macro = time.time() - t0
events = np.array(macro.events)
print(f"{len(events)} events from {n} packets ({nint_macro:.2f} interactions per packet), {wall_macro:.1f} s")

## Level 1: the matrix in six lines

In [ ]:
n_g = 4
order = np.argsort(atom.nu); group = np.empty(atom.n_lines, int)
for j, chunk in enumerate(np.array_split(order, n_g)):
    group[chunk] = j                                   # contiguous frequency groups
R = np.zeros((n_g, n_g))
for k, j in events:
    R[group[k], group[j]] += 1                         # count absorbed group -> emitted group
R /= R.sum(axis=1, keepdims=True)                      # rows: fractions of the absorbed energy
print("groups of the lines (by increasing frequency):", group); print(np.round(R, 3))
assert np.allclose(R, build_R(macro.events, *group_of_line(atom.nu, n_g)))

## Throw the atom away: transport with $R$ alone

At an interaction in group $i$, draw the exit group from row $i$, then a line inside the group from the thermal emissivity. Nothing else about the atom is used.

In [ ]:
g4, _ = group_of_line(atom.nu, 4)
t0 = time.time(); spec_R, mag_R, col_R, nint_R = spectrum(MatrixRedistribution(R, g4, emis), n, rtedu.SEEDS["ch09"] + 2); wall_R = time.time() - t0
g10, _ = group_of_line(atom.nu, 10); R10 = build_R(macro.events, g10, 10)
spec_R10, mag_R10, col_R10, _ = spectrum(MatrixRedistribution(R10, g10, emis), n, rtedu.SEEDS["ch09"] + 3)

## The scalar ε\*: the best single number

Scan ε and keep the value whose emergent spectrum is closest to the macroatom's.

In [ ]:
eps_grid = np.linspace(0, 1, 11); err_eps = []
for e in eps_grid:
    s, _, _, _ = spectrum(EpsilonRedistribution(e, emis), 1500, rtedu.SEEDS["ch09"] + 4)
    err_eps.append(float(np.abs(s - spec_macro).sum()))
eps_star = float(eps_grid[int(np.argmin(err_eps))])
t0 = time.time(); spec_eps, mag_eps, col_eps, nint_eps = spectrum(EpsilonRedistribution(eps_star, emis), n, rtedu.SEEDS["ch09"] + 5); wall_eps = time.time() - t0
err = dict(eps=float(np.abs(spec_eps - spec_macro).sum()), R4=float(np.abs(spec_R - spec_macro).sum()), R10=float(np.abs(spec_R10 - spec_macro).sum()))
noise = float(4 * np.sqrt(spec_macro * (1 - spec_macro) / n).sum())
print(f"eps* = {eps_star:.1f}; L1 errors vs macroatom: eps {err['eps']:.3f}, R(4) {err['R4']:.3f}, R(10) {err['R10']:.3f}; 4-sigma noise level {noise:.3f}")
params = dict(eps=1, R4=int(R.size), R10=int(R10.size), macro=int(atom.n_levels + atom.n_lines))
dcol = {name: {c: col[c] - col_macro[c] for c in col_macro} for name, col in (("eps", col_eps), ("R4", col_R), ("R10", col_R10))}
for name in dcol: print(name, "colour residuals:", {c: round(v, 2) for c, v in dcol[name].items()})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.arange(atom.n_lines + 1); w = 0.2
for i, (lab, s) in enumerate((("macroatom", spec_macro), (f"eps* = {eps_star:.1f}", spec_eps), ("R, 4 groups", spec_R), ("R, 10 groups", spec_R10))):
    axes[0].bar(x + (i - 1.5) * w, s, w, label=lab)
axes[0].set_xticks(x); axes[0].set_xticklabels([f"{v:.0f}" for v in nm] + ["none"], rotation=60, fontsize=7); axes[0].set_ylabel("fraction of packets"); axes[0].legend(fontsize=7); axes[0].set_title("emergent line spectrum", fontsize=9)
im = axes[1].imshow(R, cmap="viridis", vmin=0, vmax=1); axes[1].set_xlabel("emitted group j"); axes[1].set_ylabel("absorbed group i"); axes[1].set_title("R, 4 contiguous groups (rows sum to 1)", fontsize=9); plt.colorbar(im, ax=axes[1])
for i in range(n_g):
    for j in range(n_g): axes[1].text(j, i, f"{R[i, j]:.2f}", ha="center", va="center", color="w" if R[i, j] < 0.5 else "k", fontsize=8)
names = ["eps*", "R(4)", "R(10)"]
axes[2].bar(names, [err["eps"], err["R4"], err["R10"]], color=[OI["red"], OI["orange"], OI["blue"]]); axes[2].axhline(noise, color="grey", ls="--", label="4-sigma noise"); axes[2].set_ylabel("L1 error of the emergent spectrum"); axes[2].legend(fontsize=8)
axes[2].set_title("three effective models against the macroatom", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch09_matrix")

In [ ]:
results.record("ch09", dict(n=n, n_events=int(len(events)), nint_macro=nint_macro, nint_R=nint_R, nint_eps=nint_eps, n_g=n_g, group=group, R=R, R10=R10,
                            eps_star=eps_star, err=err, noise=noise, params=params,
                            colours=dict(macro=col_macro, eps=col_eps, R4=col_R, R10=col_R10), dcol=dcol, spec_macro=spec_macro, spec_eps=spec_eps, spec_R=spec_R, spec_R10=spec_R10))